# 04 — Regresión y clasificación: aprender y comprobar

Un pozo extrae fluidos del subsuelo. Aquí queremos anticipar el petróleo de **un día dentro de siete días**. Usamos registros ya conocidos de petróleo, horas de trabajo y sensores de presión/temperatura. Gas y agua también salen del pozo; no asumimos que causen cambios del petróleo.

Regresión responde cuánto. Clasificación responde si habrá una caída de al menos el 20 % respecto del día de partida, cuando ese día tiene petróleo positivo. Ninguna de las dos diagnostica fallas mecánicas.

Este cuaderno reúne los dos ejercicios con la misma preparación y fechas. Orden: comprobar datos → referencias sencillas → aprender y ajustar con días antiguos → elegir con validación → abrir la prueba final → explicar resultados. Ejecutar antes 03. Puede tardar unos minutos; los bloques informan su avance.

Las explicaciones debajo de los resultados describen la ejecución guardada de este experimento. Si cambian los datos o las decisiones y se vuelve a ejecutar, también hay que revisar esas lecturas.

## 1. Leer las reglas, sin rehacer decisiones en secreto

Comprobamos la huella de los datos. Todos los modelos de una tarea comparan las mismas fechas: no damos ventaja a un sensor por quitarle días difíciles. Los vacíos de las entradas se completan sólo dentro del entrenamiento; las respuestas nunca se completan.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (mean_absolute_error, mean_squared_error, average_precision_score,
                             precision_score, recall_score, f1_score, accuracy_score,
                             confusion_matrix, precision_recall_curve)
import plotly.express as px

ROOT = Path.cwd()
if not (ROOT / "data" / "raw").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "data" / "raw").is_dir(), "Abrir desde la raíz o notebooks/."
PROCESSED = ROOT / "data" / "processed"
archivo_reglas = PROCESSED / "reglas_modelos_7d.json"
assert archivo_reglas.is_file(), "Ejecutar primero el notebook 03."
reglas = json.loads(archivo_reglas.read_text(encoding="utf-8"))
archivo = PROCESSED / "ejemplos_modelos_7d.csv"
assert hashlib.sha256(archivo.read_bytes()).hexdigest() == reglas["sha256_ejemplos"], "Los ejemplos cambiaron: regenerar 03."
assert hashlib.sha256((ROOT / reglas["fuente"]).read_bytes()).hexdigest() == reglas["sha256_fuente"], "La preparación 02 cambió: regenerar 03."
assert hashlib.sha256((ROOT / "data/raw/Volve production data.xlsx").read_bytes()).hexdigest() == reglas["sha256_excel"]
datos = pd.read_csv(archivo, parse_dates=["DATEPRD", "FECHA_OBJETIVO"], dtype={"NPD_WELL_BORE_CODE": "string"})
grupos = reglas["grupos_numericos"]
categoricas = reglas["categoricas"]
SEED = reglas["semilla"]
for columnas in grupos.values():
    assert not any(c.startswith("Y_") or c in ["FECHA_OBJETIVO", "FILA_EXCEL_OBJETIVO"] for c in columnas)
assert (datos["FECHA_OBJETIVO"] - datos["DATEPRD"]).eq(pd.Timedelta(days=7)).all()
tareas = {"Regresión": "Y_PETROLEO_7D", "Clasificación": "Y_CAIDA_20PCT"}
partes = {}
for tarea, objetivo in tareas.items():
    base = datos.dropna(subset=[objetivo]).copy()
    if tarea == "Clasificación": base[objetivo] = base[objetivo].astype(int)
    partes[tarea] = {p: base.loc[base["PARTICION"].eq(p)].sort_values(["DATEPRD", "NPD_WELL_BORE_CODE"]).reset_index(drop=True)
                     for p in ["Entrenamiento", "Validación", "Prueba"]}
    for p, tabla in partes[tarea].items():
        assert len(tabla) > 0
        if tarea == "Clasificación": assert tabla[objetivo].nunique() == 2, f"Faltan clases en {p}."
    assert set(partes[tarea]["Prueba"][categoricas[0]]) <= set(partes[tarea]["Entrenamiento"][categoricas[0]]), "Hay pozos nuevos en prueba: revisar alcance."
display(datos.groupby("PARTICION").agg(Casos=("DATEPRD", "size"), Desde=("DATEPRD", "min"), Hasta=("DATEPRD", "max")))
print("Python", platform.python_version(), "| scikit-learn", sklearn.__version__)
RESULTADOS = ROOT / "reports" / "modelos_7d"
RESULTADOS.mkdir(parents=True, exist_ok=True)

,Casos,Desde,Hasta
PARTICION,,,
Entrenamiento,5199,2008-02-12,2014-08-22
Prueba,1711,2015-09-05,2016-09-10
Separación,70,2014-08-23,2015-09-04
Validación,1810,2014-08-30,2015-08-28


Python 3.12.10 | scikit-learn 1.9.1


**Cómo leer esta salida.** Cada fila de la tabla es una etapa, no un pozo. Casos cuenta ejemplos de regresión, Desde y Hasta indican cuándo se emitieron los pronósticos. Separación contiene días que no usamos para aprender ni evaluar para evitar cruzar respuestas futuras.

La clasificación tiene menos ejemplos porque necesita petróleo de partida positivo. Los nombres de Python y scikit-learn sólo registran con qué herramientas se ejecutó el trabajo; no son resultados del modelo. Hasta aquí cargamos y comprobamos archivos, sin entrenar.

## 2. Cómo impedimos que mire las respuestas del futuro

Dentro de los días de entrenamiento hacemos tres rondas: aprender con un tramo antiguo y comprobar con el siguiente; después ampliar el tramo aprendido. Compartimos cortes de fecha entre pozos y apartamos ejemplos cuya respuesta futura llegue al comienzo de la ronda de comprobación.

Completamos entradas vacías con la mediana de los días de aprendizaje (el valor del medio al ordenarlos), agregando marcas de faltantes. Una columna enteramente vacía en esa ronda se aparta de ese ajuste. Para modelos lineales ponemos números en escalas comparables. El nombre del pozo se convierte en categorías: nunca se interpreta como una cantidad. Si un pozo aparece por primera vez en una ronda, su categoría se ignora hasta tener ejemplos suyos.

Pipeline mantiene estos pasos juntos: cada ronda aprende su propia preparación. Ni la mediana ni las escalas se calculan usando validación o prueba.

In [2]:
def rondas_temporales(tabla, clasificacion=False):
    fechas = np.sort(tabla["DATEPRD"].unique())
    bordes = [pd.Timestamp(fechas[int(len(fechas) * x)]) for x in [0.50, 2/3, 5/6]]
    finales = bordes[1:] + [tabla["DATEPRD"].max() + pd.Timedelta(days=1)]
    rondas = []
    for inicio, fin in zip(bordes, finales):
        aprender = np.flatnonzero((tabla["FECHA_OBJETIVO"] < inicio).to_numpy())
        comprobar = np.flatnonzero(((tabla["DATEPRD"] >= inicio) & (tabla["DATEPRD"] < fin)).to_numpy())
        assert len(aprender) and len(comprobar)
        assert tabla.iloc[aprender]["FECHA_OBJETIVO"].max() < tabla.iloc[comprobar]["DATEPRD"].min()
        if clasificacion:
            assert tabla.iloc[aprender]["Y_CAIDA_20PCT"].nunique() == 2
            assert tabla.iloc[comprobar]["Y_CAIDA_20PCT"].nunique() == 2
        rondas.append((aprender, comprobar))
    return rondas

def preparar_modelo(columnas, estimador):
    numeros = Pipeline([("completar", SimpleImputer(strategy="median", add_indicator=True)),
                        ("escala", StandardScaler())])
    preparacion = ColumnTransformer([
        ("numeros", numeros, columnas),
        ("pozo", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categoricas),
    ])
    return Pipeline([("preparar", preparacion), ("modelo", estimador)])

rondas = {t: rondas_temporales(partes[t]["Entrenamiento"], t == "Clasificación") for t in tareas}
filas_rondas = []
for tarea in tareas:
    tabla = partes[tarea]["Entrenamiento"]
    for i, (a, b) in enumerate(rondas[tarea], 1):
        filas_rondas.append({"Tarea": tarea, "Ronda": i, "Aprender": len(a), "Comprobar": len(b),
                             "Última respuesta aprendida": tabla.iloc[a]["FECHA_OBJETIVO"].max(),
                             "Primera fecha comprobada": tabla.iloc[b]["DATEPRD"].min()})
display(pd.DataFrame(filas_rondas))

,Tarea,Ronda,Aprender,Comprobar,Última respuesta aprendida,Primera fecha comprobada
0,Regresión,1,2206,740,2011-04-01,2011-04-02
1,Regresión,2,2946,772,2012-06-09,2012-06-10
2,Regresión,3,3711,1467,2013-08-08,2013-08-09
3,Clasificación,1,1953,711,2011-03-14,2011-03-15
4,Clasificación,2,2664,723,2012-06-08,2012-06-09
5,Clasificación,3,3380,1371,2013-08-18,2013-08-19


**Qué significa esta tabla.** Cada ronda es un pequeño ensayo dentro del entrenamiento: aprender con días antiguos y comprobar con días posteriores. Aprender y Comprobar indican cuántos ejemplos usamos. En cada fila, Última respuesta aprendida debe ser anterior a Primera fecha comprobada.

Así la computadora no estudia una respuesta que todavía no existía cuando empezó el ensayo. Las tres rondas sirven para ajustar los modelos; no reemplazan el examen final. Este bloque prepara las reglas, todavía no muestra qué modelo funciona mejor.

## 3. Referencias: ¿qué pasa si hacemos algo muy sencillo?

Para regresión usamos persistencia: dentro de siete días habrá lo mismo que hoy. Para clasificación usamos una regla que siempre elige la clase más frecuente del entrenamiento; su puntuación de caída es la proporción de caídas de esos días. Son referencias, no modelos útiles por definición.

MAE mide el tamaño promedio del error de cantidad; RMSE da más peso a errores grandes. Menor es mejor. En clasificación, average precision (AP) resume si las caídas reales quedan arriba cuando ordenamos por riesgo; mayor es mejor, pero depende de cuántas caídas haya. Una puntuación constante tiene AP igual a la proporción de caídas evaluadas. No basta con acertar muchos días sin caída.

In [3]:
def metricas_regresion(real, pred):
    return {"MAE": mean_absolute_error(real, pred), "RMSE": float(np.sqrt(mean_squared_error(real, pred)))}

def metricas_clasificacion(real, prob, umbral=0.5):
    pred = np.asarray(prob) >= umbral
    return {"AP": average_precision_score(real, prob) if pd.Series(real).nunique() == 2 else np.nan,
            "Precision": precision_score(real, pred, zero_division=0),
            "Recall": recall_score(real, pred, zero_division=0), "F1": f1_score(real, pred, zero_division=0),
            "Accuracy": accuracy_score(real, pred), "Prevalencia": float(np.mean(real))}

referencia_clase = DummyClassifier(strategy="prior")
tr_clase = partes["Clasificación"]["Entrenamiento"]
referencia_clase.fit(tr_clase[["PETROLEO_T"]], tr_clase["Y_CAIDA_20PCT"])
referencias_validacion = []
for tarea in tareas:
    val = partes[tarea]["Validación"]
    met = metricas_regresion(val["Y_PETROLEO_7D"], val["PETROLEO_T"]) if tarea == "Regresión" else metricas_clasificacion(val["Y_CAIDA_20PCT"], referencia_clase.predict_proba(val[["PETROLEO_T"]])[:, 1])
    referencias_validacion.append({"Tarea": tarea, **met})
display(pd.DataFrame(referencias_validacion))

,Tarea,MAE,RMSE,AP,Precision,Recall,F1,Accuracy,Prevalencia
0,Regresión,95.485398,214.708565,NaN,NaN,NaN,NaN,NaN,NaN
1,Clasificación,NaN,NaN,0.126309,0.0,0.0,0.0,0.873691,0.126309


**Qué nos dicen las referencias.** En regresión, repetir el petróleo de hoy tiene un error promedio de aproximadamente 95,49 en los días de validación. Esa es la marca que intentamos mejorar; no significa equivocarnos un 95,49 %, porque MAE conserva la unidad de la cantidad, todavía pendiente de confirmar.

En clasificación, la referencia dice siempre sin caída. Acierta cerca del 87,4 % porque las caídas son poco frecuentes, pero no encuentra ninguna: recall y F1 son cero. Su AP es aproximadamente 0,126. Esto muestra por qué mirar sólo el porcentaje de aciertos puede engañar.

## 4. Regresión: probar modelos y ajustar su complejidad

Ridge aprende una combinación de pistas y limita cuánto depende de cada una. Un bosque combina árboles que hacen preguntas sobre las pistas. Probamos pocos ajustes: fuerza del límite de Ridge; profundidad y mínimo de ejemplos por hoja del bosque. Todos usan semilla fija para poder repetir el experimento.

Para cada grupo A–E buscamos el menor MAE promedio en las tres rondas antiguas. Después comparamos esos candidatos con validación. Miramos también el error de entrenamiento: una diferencia grande puede señalar sobreajuste, aunque también puede reflejar cambios del pozo. No garantiza que optimizar ajustes elimine el problema.

No eliminamos fechas porque un sensor esté vacío. No recortamos predicciones negativas de Ridge: se cuentan como error y se informa cuántas produjo.

In [4]:
resultados_cv = []
candidatos = {}
busquedas = {}

def ajustar_tarea(tarea):
    tabla = partes[tarea]["Entrenamiento"]
    val = partes[tarea]["Validación"]
    objetivo = tareas[tarea]
    if tarea == "Regresión":
        familias = {
            "Ridge": (Ridge(), {"modelo__alpha": [1.0, 100.0, 1000.0]}),
            "Bosque": (RandomForestRegressor(n_estimators=80, random_state=SEED, n_jobs=1),
                       [{"modelo__max_depth": [4], "modelo__min_samples_leaf": [20], "modelo__max_features": [0.8]},
                        {"modelo__max_depth": [8], "modelo__min_samples_leaf": [10], "modelo__max_features": [0.8]}]),
        }
        scoring = "neg_mean_absolute_error"
    else:
        familias = {
            "Logística": (LogisticRegression(max_iter=3000, solver="lbfgs"), {"modelo__C": [0.01, 0.1, 1.0]}),
            "Bosque": (RandomForestClassifier(n_estimators=80, random_state=SEED, n_jobs=1),
                       [{"modelo__max_depth": [4], "modelo__min_samples_leaf": [20], "modelo__max_features": [0.8]},
                        {"modelo__max_depth": [8], "modelo__min_samples_leaf": [10], "modelo__max_features": [0.8]}]),
        }
        scoring = "average_precision"
    filas = []
    for grupo, columnas in grupos.items():
        entradas = columnas + categoricas
        for familia, (estimador, opciones) in familias.items():
            busqueda = GridSearchCV(preparar_modelo(columnas, estimador), opciones, cv=rondas[tarea],
                                    scoring=scoring, refit=True, return_train_score=True, n_jobs=1, error_score="raise")
            busqueda.fit(tabla[entradas], tabla[objetivo])
            modelo = busqueda.best_estimator_
            clave = (tarea, grupo, familia)
            candidatos[clave] = modelo
            busquedas[clave] = busqueda
            resumen_cv = pd.DataFrame(busqueda.cv_results_)
            for _, r in resumen_cv.iterrows():
                resultados_cv.append({"Tarea": tarea, "Grupo": grupo, "Familia": familia,
                                      "Parámetros": json.dumps(r["params"], ensure_ascii=False),
                                      "Puntuación CV": r["mean_test_score"], "Desvío CV": r["std_test_score"],
                                      "Puntuación entrenamiento CV": r["mean_train_score"], "Orden": int(r["rank_test_score"])})
            if tarea == "Regresión":
                p_train = modelo.predict(tabla[entradas]); p_val = modelo.predict(val[entradas])
                fila = {"MAE entrenamiento": mean_absolute_error(tabla[objetivo], p_train),
                        **metricas_regresion(val[objetivo], p_val), "MAE CV": -busqueda.best_score_}
            else:
                p_train = modelo.predict_proba(tabla[entradas])[:, 1]; p_val = modelo.predict_proba(val[entradas])[:, 1]
                fila = {"AP entrenamiento": average_precision_score(tabla[objetivo], p_train),
                        **metricas_clasificacion(val[objetivo], p_val), "AP CV": busqueda.best_score_}
            filas.append({"Tarea": tarea, "Grupo": grupo, "Familia": familia,
                          "Parámetros": json.dumps(busqueda.best_params_, ensure_ascii=False), **fila})
            print(tarea, grupo, familia, "completado")
    return pd.DataFrame(filas)

validacion_reg = ajustar_tarea("Regresión")
display(validacion_reg.sort_values("MAE"))
fig_reg = px.bar(validacion_reg, x="Grupo", y="MAE", color="Familia", barmode="group",
                 title="Regresión: error en los días de práctica (menor es mejor)", template="plotly_white")
fig_reg.add_hline(y=referencias_validacion[0]["MAE"], line_dash="dash", annotation_text="Referencia: mismo petróleo que hoy")
fig_reg.show()

Regresión A_Historia Ridge completado
Regresión A_Historia Bosque completado
Regresión B_Horas Ridge completado
Regresión B_Horas Bosque completado
Regresión C_Presiones Ridge completado
Regresión C_Presiones Bosque completado
Regresión D_Sensores Ridge completado
Regresión D_Sensores Bosque completado
Regresión E_Gas_agua Ridge completado
Regresión E_Gas_agua Bosque completado


,Tarea,Grupo,Familia,Parámetros,MAE entrenamiento,MAE,RMSE,MAE CV
0,Regresión,A_Historia,Ridge,"{""modelo__alpha"": 1.0}",357.199186,127.039816,207.298410,254.223414
3,Regresión,B_Horas,Bosque,"{""modelo__max_depth"": 8, ""modelo__max_features...",294.209233,143.929012,226.262691,361.798449
2,Regresión,B_Horas,Ridge,"{""modelo__alpha"": 1.0}",362.589019,146.040262,234.439659,222.771464
1,Regresión,A_Historia,Bosque,"{""modelo__max_depth"": 8, ""modelo__max_features...",296.034252,146.204122,227.128485,358.376638
8,Regresión,E_Gas_agua,Ridge,"{""modelo__alpha"": 100.0}",356.812161,199.373067,299.861087,319.181059
6,Regresión,D_Sensores,Ridge,"{""modelo__alpha"": 100.0}",357.687166,199.452329,296.802607,280.306365
4,Regresión,C_Presiones,Ridge,"{""modelo__alpha"": 1.0}",355.069635,204.801966,320.225032,225.834121
5,Regresión,C_Presiones,Bosque,"{""modelo__max_depth"": 8, ""modelo__max_features...",272.960073,234.712887,403.717623,378.664420
7,Regresión,D_Sensores,Bosque,"{""modelo__max_depth"": 8, ""modelo__max_features...",263.075612,269.131925,395.052990,405.413603
9,Regresión,E_Gas_agua,Bosque,"{""modelo__max_depth"": 8, ""modelo__max_features...",258.410781,274.782068,393.638100,427.038165


**Cómo leer la tabla y las barras de regresión.** Cada grupo A–E agrega pistas al anterior y los colores distinguen Ridge y bosque. MAE es el error promedio en los días de práctica: menor es mejor. MAE CV viene de las tres rondas antiguas; MAE entrenamiento viene de días ya estudiados. No son las mismas fechas ni necesariamente la misma escala de producción: sus diferencias no prueban por sí solas sobreajuste.

En el gráfico queremos barras por debajo de la línea punteada de la referencia. No ocurre en esta ejecución: el mejor supervisado es Ridge con historia, con MAE de 127,04, frente a 95,49 de repetir hoy. Agregar sensores y gas/agua no lo mejora en esta comparación. Conservamos la regla sencilla; no significa que esos datos nunca puedan servir en otro planteo.

## 5. Clasificación: encontrar caídas sin mirar el futuro

La regresión logística devuelve una puntuación entre 0 y 1 para la caída. El bosque también produce una puntuación; todavía no comprobamos si esos números son probabilidades bien calibradas. No son una garantía.

Buscamos ajustes que ordenen mejor los casos de caída usando AP en las tres rondas antiguas. En la primera tabla precision/recall/F1 usan el corte 0,5 sólo para comparar. El próximo bloque elige el punto de alarma en validación. No usamos accuracy para elegir porque podría ocultar caídas cuando hay muchos días sin ellas.

In [5]:
validacion_clf = ajustar_tarea("Clasificación")
display(validacion_clf.sort_values("AP", ascending=False))
fig_clf = px.bar(validacion_clf, x="Grupo", y="AP", color="Familia", barmode="group",
                 title="Clasificación: ordenar caídas en práctica (mayor AP es mejor)", template="plotly_white")
fig_clf.add_hline(y=referencias_validacion[1]["AP"], line_dash="dash", annotation_text="Referencia sin distinguir días")
fig_clf.show()

Clasificación A_Historia Logística completado
Clasificación A_Historia Bosque completado
Clasificación B_Horas Logística completado
Clasificación B_Horas Bosque completado
Clasificación C_Presiones Logística completado
Clasificación C_Presiones Bosque completado
Clasificación D_Sensores Logística completado
Clasificación D_Sensores Bosque completado
Clasificación E_Gas_agua Logística completado
Clasificación E_Gas_agua Bosque completado


,Tarea,Grupo,Familia,Parámetros,AP entrenamiento,AP,Precision,Recall,F1,Accuracy,Prevalencia,AP CV
9,Clasificación,E_Gas_agua,Bosque,"{""modelo__max_depth"": 4, ""modelo__max_features...",0.428944,0.340755,0.0,0.0,0.0,0.873691,0.126309,0.164687
0,Clasificación,A_Historia,Logística,"{""modelo__C"": 0.1}",0.188654,0.300653,0.0,0.0,0.0,0.873691,0.126309,0.146474
6,Clasificación,D_Sensores,Logística,"{""modelo__C"": 0.1}",0.203393,0.300267,0.0,0.0,0.0,0.873691,0.126309,0.156580
7,Clasificación,D_Sensores,Bosque,"{""modelo__max_depth"": 4, ""modelo__max_features...",0.427611,0.299426,0.0,0.0,0.0,0.873691,0.126309,0.195574
8,Clasificación,E_Gas_agua,Logística,"{""modelo__C"": 0.1}",0.209966,0.298508,0.0,0.0,0.0,0.873691,0.126309,0.164963
4,Clasificación,C_Presiones,Logística,"{""modelo__C"": 1.0}",0.193500,0.273388,0.0,0.0,0.0,0.873691,0.126309,0.138045
5,Clasificación,C_Presiones,Bosque,"{""modelo__max_depth"": 4, ""modelo__max_features...",0.394239,0.238543,0.0,0.0,0.0,0.873691,0.126309,0.183330
3,Clasificación,B_Horas,Bosque,"{""modelo__max_depth"": 4, ""modelo__max_features...",0.353798,0.232457,0.0,0.0,0.0,0.873691,0.126309,0.184935
1,Clasificación,A_Historia,Bosque,"{""modelo__max_depth"": 4, ""modelo__max_features...",0.352017,0.231908,0.0,0.0,0.0,0.873691,0.126309,0.197325
2,Clasificación,B_Horas,Logística,"{""modelo__C"": 0.01}",0.185363,0.212806,0.0,0.0,0.0,0.873691,0.126309,0.140638


**Cómo leer la tabla y las barras de clasificación.** AP indica qué tan bien quedan las caídas reales entre los casos de mayor puntuación. Aquí una barra más alta es mejor; la línea punteada es la referencia de aproximadamente 0,126. El mayor valor es 0,341 para el bosque con el grupo E, que incluye gas y agua conocidos.

AP de 0,341 no significa 34,1 % de aciertos. En la tabla, precision, recall y F1 son cero porque con el corte inicial de 0,50 ninguno de estos modelos genera alarmas en validación. Pueden ordenar los riesgos de forma diferente aunque ninguna puntuación alcance 0,50. La siguiente celda elige otro corte con validación, sin tocar prueba.

Ganar con el grupo E no demuestra que cada sensor sea necesario ni que el gas cause cambios: estamos comparando grupos y modelos completos.

## 6. Cerrar la selección antes de abrir el examen

Elegimos por MAE de validación para regresión y AP de validación para clasificación. En empate preferimos grupos con menos pistas y después el modelo lineal. Si la referencia sencilla empata o supera al mejor candidato, conservamos la referencia. También guardamos el mejor candidato supervisado para mostrar si la tarea exigía entrenar un modelo aunque la referencia ganara.

En clasificación elegimos entre cortes de 0,10 a 0,90 el que da mayor F1 en validación. En empate, el corte mayor genera menos alarmas. Es un criterio de ejercicio, no una decisión basada en costos reales. No lo ajustamos con prueba. Si gana la referencia de clase mayoritaria no buscamos un corte para ella.

Los modelos se mantienen ajustados sólo con entrenamiento: no se vuelven a entrenar con validación. Así el punto de alarma corresponde al mismo modelo que evaluaremos. Este cuaderno no promete que un bosque gane; también es válido que la referencia sea mejor.

In [6]:
mejor_reg = validacion_reg.sort_values(["MAE", "Grupo"], kind="stable").iloc[0]
mejor_clf = validacion_clf.sort_values(["AP", "Grupo"], ascending=[False, True], kind="stable").iloc[0]
ganadores = {
    "Regresión": {"grupo": mejor_reg["Grupo"], "familia": mejor_reg["Familia"], "referencia": referencias_validacion[0]["MAE"] <= mejor_reg["MAE"]},
    "Clasificación": {"grupo": mejor_clf["Grupo"], "familia": mejor_clf["Familia"], "referencia": referencias_validacion[1]["AP"] >= mejor_clf["AP"]},
}
elegidos = {t: candidatos[(t, g["grupo"], g["familia"])] for t, g in ganadores.items()}
val = partes["Clasificación"]["Validación"]
columnas_clf = grupos[ganadores["Clasificación"]["grupo"]] + categoricas
p_val = elegidos["Clasificación"].predict_proba(val[columnas_clf])[:, 1]
tabla_umbrales = pd.DataFrame([{"Umbral": float(u), **metricas_clasificacion(val["Y_CAIDA_20PCT"], p_val, u)} for u in np.linspace(0.1, 0.9, 9)])
umbral_candidato = float(tabla_umbrales.sort_values(["F1", "Umbral"], ascending=[False, False]).iloc[0]["Umbral"])
umbral_final = 0.5 if ganadores["Clasificación"]["referencia"] else umbral_candidato
display(tabla_umbrales)
for tarea, ganador in ganadores.items():
    display(Markdown(f"**{tarea}:** mejor candidato supervisado: {ganador['familia']} con {ganador['grupo']}. "
                     + ("La referencia gana o empata: queda como elección final." if ganador['referencia'] else "El candidato supera a la referencia en validación y queda elegido.")))
print("Corte de alarma final:", umbral_final)

def predecir(tarea, tabla, usar_referencia=False):
    if usar_referencia:
        return tabla["PETROLEO_T"].to_numpy() if tarea == "Regresión" else referencia_clase.predict_proba(tabla[["PETROLEO_T"]])[:, 1]
    columnas = grupos[ganadores[tarea]["grupo"]] + categoricas
    modelo = elegidos[tarea]
    return modelo.predict(tabla[columnas]) if tarea == "Regresión" else modelo.predict_proba(tabla[columnas])[:, 1]

,Umbral,AP,Precision,Recall,F1,Accuracy,Prevalencia
0,0.1,0.340755,0.133125,0.880829,0.231293,0.260471,0.126309
1,0.2,0.340755,0.294872,0.357513,0.323185,0.810864,0.126309
2,0.3,0.340755,0.500000,0.186528,0.271698,0.873691,0.126309
3,0.4,0.340755,1.000000,0.005181,0.010309,0.874346,0.126309
4,0.5,0.340755,0.000000,0.000000,0.000000,0.873691,0.126309
5,0.6,0.340755,0.000000,0.000000,0.000000,0.873691,0.126309
6,0.7,0.340755,0.000000,0.000000,0.000000,0.873691,0.126309
7,0.8,0.340755,0.000000,0.000000,0.000000,0.873691,0.126309
8,0.9,0.340755,0.000000,0.000000,0.000000,0.873691,0.126309


**Regresión:** mejor candidato supervisado: Ridge con A_Historia. La referencia gana o empata: queda como elección final.

**Clasificación:** mejor candidato supervisado: Bosque con E_Gas_agua. El candidato supera a la referencia en validación y queda elegido.

Corte de alarma final: 0.2


**Qué decisión salió de este bloque.** Para cantidades elegimos la referencia sencilla; Ridge queda como el mejor supervisado comparado, pero no como ganador. Para caídas elegimos el bosque con grupo E.

Umbral es el punto a partir del cual avisamos caída. Con 0,10 el modelo encuentra muchas caídas de validación, pero sólo cerca de 13 de cada 100 alarmas son correctas. Con 0,20 encuentra alrededor del 36 % de las caídas y cerca de 29 de cada 100 alarmas son correctas; ese corte da el mejor F1 de los probados, aproximadamente 0,323. Con 0,50 no avisa ninguna.

El 0,20 es una puntuación usada para activar la alarma; no garantiza un 20 % de probabilidad real. Tampoco es el mismo 20 % que define cuánto debe bajar el petróleo. Son dos números con funciones distintas. Dejamos las decisiones cerradas antes de mirar prueba.

## 7. Examen final: cantidades, caídas y resultados por pozo

Recién aquí calculamos predicciones de prueba, sin cambiar las decisiones anteriores. Mostramos referencia, mejor supervisado y elección final: si coinciden, la repetición permite saber qué representa cada fila.

Precision pregunta cuántas alarmas eran caídas reales. Recall pregunta cuántas caídas encontramos. F1 combina ambas. La matriz de confusión separa aciertos, falsas alarmas y caídas omitidas. Si un pozo no tiene ambas clases, AP queda vacía: no se interpreta como cero.

Los resultados sólo describen filas que pasaron las reglas de calidad y el requisito del objetivo. La clasificación no incluye días de partida con petróleo cero. Los días cercanos están relacionados; estos números no son intervalos de confianza ni pruebas de causalidad.

In [7]:
metricas_prueba = []
por_pozo = []
predicciones = {}
for tarea, objetivo in tareas.items():
    tabla = partes[tarea]["Prueba"]
    base = tabla[["NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "DATEPRD", "FECHA_OBJETIVO", "FILA_EXCEL", "FILA_EXCEL_OBJETIVO", objetivo]].copy()
    base["REFERENCIA"] = predecir(tarea, tabla, True)
    base["SUPERVISADO"] = predecir(tarea, tabla)
    base["ELEGIDO"] = base["REFERENCIA"] if ganadores[tarea]["referencia"] else base["SUPERVISADO"]
    for nombre in ["REFERENCIA", "SUPERVISADO", "ELEGIDO"]:
        umbral = 0.5 if nombre == "REFERENCIA" or (nombre == "ELEGIDO" and ganadores[tarea]["referencia"]) else umbral_candidato
        met = metricas_regresion(base[objetivo], base[nombre]) if tarea == "Regresión" else metricas_clasificacion(base[objetivo], base[nombre], umbral)
        metricas_prueba.append({"Tarea": tarea, "Modelo": nombre, "Casos": len(base), **met})
        for pozo, g in base.groupby("NPD_WELL_BORE_NAME"):
            met_pozo = metricas_regresion(g[objetivo], g[nombre]) if tarea == "Regresión" else metricas_clasificacion(g[objetivo], g[nombre], umbral)
            por_pozo.append({"Tarea": tarea, "Modelo": nombre, "Pozo": pozo, "Casos": len(g), **met_pozo})
    if tarea == "Clasificación": base["ALARMA_ELEGIDA"] = (base["ELEGIDO"] >= umbral_final).astype(int)
    predicciones[tarea] = base
tabla_prueba = pd.DataFrame(metricas_prueba)
tabla_pozos = pd.DataFrame(por_pozo)
display(tabla_prueba)
display(tabla_pozos)
for tarea, objetivo in tareas.items():
    conocidas = set(datos["NPD_WELL_BORE_NAME"])
    presentes = set(partes[tarea]["Prueba"]["NPD_WELL_BORE_NAME"])
    print(tarea, "| Pozos sin casos en prueba:", sorted(conocidas - presentes))
print("Predicciones negativas del candidato de regresión:", int((predicciones["Regresión"]["SUPERVISADO"] < 0).sum()))

,Tarea,Modelo,Casos,MAE,RMSE,AP,Precision,Recall,F1,Accuracy,Prevalencia
0,Regresión,REFERENCIA,1711,53.187510,138.028284,NaN,NaN,NaN,NaN,NaN,NaN
1,Regresión,SUPERVISADO,1711,92.628279,138.968114,NaN,NaN,NaN,NaN,NaN,NaN
2,Regresión,ELEGIDO,1711,53.187510,138.028284,NaN,NaN,NaN,NaN,NaN,NaN
3,Clasificación,REFERENCIA,1391,NaN,NaN,0.117901,0.000000,0.00000,0.000000,0.882099,0.117901
4,Clasificación,SUPERVISADO,1391,NaN,NaN,0.214965,0.321429,0.27439,0.296053,0.846154,0.117901
5,Clasificación,ELEGIDO,1391,NaN,NaN,0.214965,0.321429,0.27439,0.296053,0.846154,0.117901


,Tarea,Modelo,Pozo,Casos,MAE,RMSE,AP,Precision,Recall,F1,Accuracy,Prevalencia
0,Regresión,REFERENCIA,15/9-F-1 C,223,116.825426,178.842514,NaN,NaN,NaN,NaN,NaN,NaN
1,Regresión,REFERENCIA,15/9-F-11,372,99.463710,245.662676,NaN,NaN,NaN,NaN,NaN,NaN
2,Regresión,REFERENCIA,15/9-F-12,372,20.212419,35.181189,NaN,NaN,NaN,NaN,NaN,NaN
3,Regresión,REFERENCIA,15/9-F-14,372,11.037043,21.082397,NaN,NaN,NaN,NaN,NaN,NaN
4,Regresión,REFERENCIA,15/9-F-15 D,372,43.888333,80.138292,NaN,NaN,NaN,NaN,NaN,NaN
5,Regresión,SUPERVISADO,15/9-F-1 C,223,159.205458,198.294455,NaN,NaN,NaN,NaN,NaN,NaN
6,Regresión,SUPERVISADO,15/9-F-11,372,96.774011,199.774503,NaN,NaN,NaN,NaN,NaN,NaN
7,Regresión,SUPERVISADO,15/9-F-12,372,97.540223,103.675346,NaN,NaN,NaN,NaN,NaN,NaN
8,Regresión,SUPERVISADO,15/9-F-14,372,86.609572,89.319362,NaN,NaN,NaN,NaN,NaN,NaN
9,Regresión,SUPERVISADO,15/9-F-15 D,372,49.678796,81.349823,NaN,NaN,NaN,NaN,NaN,NaN


Regresión | Pozos sin casos en prueba: []
Clasificación | Pozos sin casos en prueba: []
Predicciones negativas del candidato de regresión: 0


**Cómo leer las tablas del examen.** REFERENCIA es la regla sencilla; SUPERVISADO, el mejor candidato elegido en práctica; ELEGIDO, lo que decidimos usar. Algunas filas repiten números porque ELEGIDO puede ser la referencia o el supervisado. Los casilleros vacíos corresponden a medidas que no aplican a esa tarea.

En cantidades, la referencia tiene MAE 53,19 y Ridge 92,63 en prueba: sigue ganando la referencia. Que el error sea menor que en validación no demuestra que el modelo haya aprendido más: estamos mirando otro período.

En caídas, el elegido tiene AP 0,215 frente a 0,118 de la referencia. Detecta aproximadamente 27 de cada 100 caídas y cerca de 32 de cada 100 alarmas son correctas. Es una mejora para ordenar casos, pero todavía deja muchas caídas sin detectar.

La tabla por pozo importa: 44 de las 45 caídas detectadas son de F-1 C y una de F-11. Con el corte elegido no detecta caídas de F-12, F-14 ni F-15 D. El promedio general oculta esa diferencia. Los cinco pozos sí tienen ejemplos de prueba; una lista vacía en Pozos sin casos no significa cero pozos evaluados.

### Un dibujo para ver errores de cantidad

Cada punto es un día de prueba. Comparamos petróleo real con lo predicho. En la línea diagonal acertaríamos exactamente; cuanto más lejos, mayor error. El color indica el pozo. Las unidades diarias siguen pendientes de confirmar.

In [8]:
p_reg = predicciones["Regresión"]
fig_cantidades = px.scatter(p_reg, x="Y_PETROLEO_7D", y="ELEGIDO", color="NPD_WELL_BORE_NAME", hover_data=["DATEPRD", "FECHA_OBJETIVO"],
                           labels={"Y_PETROLEO_7D": "Petróleo real (unidad pendiente)", "ELEGIDO": "Petróleo estimado (unidad pendiente)", "NPD_WELL_BORE_NAME": "Pozo"},
                           title="Prueba final: cuánto se aleja la estimación del dato real", template="plotly_white", opacity=0.5)
limite = float(max(p_reg["Y_PETROLEO_7D"].max(), p_reg["ELEGIDO"].max()))
fig_cantidades.add_shape(type="line", x0=0, y0=0, x1=limite, y1=limite, line=dict(dash="dash", color="gray"))
fig_cantidades.show()

**Qué nos dice este gráfico.** De izquierda a derecha está el petróleo que realmente se observó; la altura muestra lo que estimamos. Sobre la diagonal acertaríamos exactamente. Un punto por encima significa que estimamos de más; por debajo, de menos. El color sólo identifica al pozo.

El gráfico muestra ELEGIDO, que en regresión es repetir el valor de hoy, no Ridge. Resume 1.711 ejemplos de prueba y su error promedio es 53,19. Ese número no es un porcentaje ni un máximo: algunos días tienen errores mucho mayores. No alcanza con que muchos puntos se vean cerca de la línea para afirmar que el método funciona igual en todos los pozos.

### Una tabla dibujada para las caídas

Las filas indican lo que realmente pasó y las columnas, lo que avisamos. Arriba a la derecha están las falsas alarmas; abajo a la izquierda, las caídas que se escaparon. Una alarma significa la caída definida en este ejercicio, no una falla mecánica.

In [9]:
p_clf = predicciones["Clasificación"]
matriz = confusion_matrix(p_clf["Y_CAIDA_20PCT"], p_clf["ALARMA_ELEGIDA"], labels=[0, 1])
fig_confusion = px.imshow(matriz, text_auto=True, x=["Sin alarma", "Alarma de caída"], y=["Sin caída real", "Caída real"],
                          labels=dict(x="Lo que avisamos", y="Lo que pasó", color="Días"),
                          title="Clasificación elegida: aciertos y errores en prueba", color_continuous_scale="Blues")
fig_confusion.show()

**Leamos los cuatro casilleros.** Arriba a la izquierda hay 1.132 días sin caída en los que no avisamos. Arriba a la derecha, 95 falsas alarmas. Abajo a la izquierda, 119 caídas que se escaparon. Abajo a la derecha, 45 caídas detectadas.

En total hubo 164 caídas y encontramos 45. Lanzamos 140 alarmas y sólo 45 coincidieron con una caída real. El casillero grande de días sin caída no debe distraernos: todavía se escapan la mayoría de las caídas. Los colores muestran cantidad de días, no probabilidades ni causas.

## 8. Conclusiones calculadas y archivos para revisar

La comparación de grupos de validación orienta qué pistas agregan valor; no demuestra un efecto físico. Una mejora de grupo puede deberse a varias variables y cambia entre modelos. Como cada familia ajusta sus parámetros, la comparación es de procedimientos completos, no una importancia individual.

Guardamos métricas, búsqueda de ajustes, fechas y predicciones con fila Excel. No publicamos ni desplegamos un servicio. El siguiente trabajo será revisar errores, probar estabilidad y después abordar incertidumbre y clustering; estos dos temas no se resuelven aquí.

In [10]:
for tarea, metrica, menor in [("Regresión", "MAE", True), ("Clasificación", "AP", False)]:
    tabla = tabla_prueba.loc[tabla_prueba["Tarea"].eq(tarea)].set_index("Modelo")
    ref = float(tabla.loc["REFERENCIA", metrica]); elegido = float(tabla.loc["ELEGIDO", metrica])
    mejora = elegido < ref if menor else elegido > ref
    display(Markdown(f"**{tarea}, examen final:** {metrica} elegido = {elegido:.3f}; referencia = {ref:.3f}. "
                     + ("La elección mejora esa medida en este período." if mejora else "La elección no mejora esa medida frente a la referencia en este período.")
                     + " No cambiamos el modelo después de ver este resultado."))

# Explicar el resultado de clasificación en cantidades de días, no sólo siglas.
real = predicciones["Clasificación"]["Y_CAIDA_20PCT"].astype(int)
alarma = predicciones["Clasificación"]["ALARMA_ELEGIDA"].astype(int)
caidas = int(real.sum())
encontradas = int(((real == 1) & (alarma == 1)).sum())
falsas = int(((real == 0) & (alarma == 1)).sum())
resumen_clase = (f"De {caidas} caídas reales de prueba, encontramos {encontradas} y se escaparon {caidas-encontradas}. "
                 f"Hubo {falsas} falsas alarmas. Una mejor AP no significa que encontremos todas las caídas.")
display(Markdown("**En palabras simples:** " + resumen_clase))

validacion_reg.to_csv(RESULTADOS / "validacion_regresion.csv", index=False)
validacion_clf.to_csv(RESULTADOS / "validacion_clasificacion.csv", index=False)
pd.DataFrame(resultados_cv).to_csv(RESULTADOS / "busqueda_hiperparametros.csv", index=False)
tabla_umbrales.to_csv(RESULTADOS / "umbrales_validacion.csv", index=False)
tabla_prueba.to_csv(RESULTADOS / "metricas_prueba.csv", index=False)
tabla_pozos.to_csv(RESULTADOS / "metricas_por_pozo.csv", index=False)
for tarea, pred in predicciones.items():
    nombre = "regresion" if tarea == "Regresión" else "clasificacion"
    pred.to_csv(RESULTADOS / f"predicciones_{nombre}.csv", index=False, date_format="%Y-%m-%d")
seleccion = {t: {**g, "referencia": bool(g["referencia"]), "parametros_supervisado": busquedas[(t, g["grupo"], g["familia"])].best_params_} for t, g in ganadores.items()}
seleccion.update({"umbral_final": umbral_final, "umbral_supervisado": umbral_candidato,
                  "sha256_ejemplos": reglas["sha256_ejemplos"], "python": platform.python_version(), "sklearn": sklearn.__version__,
                  "entrenamiento_final": "Sólo entrenamiento; validación no se incorpora al ajuste final", "semilla": SEED})
(RESULTADOS / "seleccion_modelos.json").write_text(json.dumps(seleccion, ensure_ascii=False, indent=2), encoding="utf-8")
from html import escape
lecturas_resultados = {2: '**Cómo leer esta salida.** Cada fila de la tabla es una etapa, no un pozo. Casos cuenta ejemplos de regresión, Desde y Hasta indican cuándo se emitieron los pronósticos. Separación contiene días que no usamos para aprender ni evaluar para evitar cruzar respuestas futuras.\n\nLa clasificación tiene menos ejemplos porque necesita petróleo de partida positivo. Los nombres de Python y scikit-learn sólo registran con qué herramientas se ejecutó el trabajo; no son resultados del modelo. Hasta aquí cargamos y comprobamos archivos, sin entrenar.', 4: '**Qué significa esta tabla.** Cada ronda es un pequeño ensayo dentro del entrenamiento: aprender con días antiguos y comprobar con días posteriores. Aprender y Comprobar indican cuántos ejemplos usamos. En cada fila, Última respuesta aprendida debe ser anterior a Primera fecha comprobada.\n\nAsí la computadora no estudia una respuesta que todavía no existía cuando empezó el ensayo. Las tres rondas sirven para ajustar los modelos; no reemplazan el examen final. Este bloque prepara las reglas, todavía no muestra qué modelo funciona mejor.', 6: '**Qué nos dicen las referencias.** En regresión, repetir el petróleo de hoy tiene un error promedio de aproximadamente 95,49 en los días de validación. Esa es la marca que intentamos mejorar; no significa equivocarnos un 95,49 %, porque MAE conserva la unidad de la cantidad, todavía pendiente de confirmar.\n\nEn clasificación, la referencia dice siempre sin caída. Acierta cerca del 87,4 % porque las caídas son poco frecuentes, pero no encuentra ninguna: recall y F1 son cero. Su AP es aproximadamente 0,126. Esto muestra por qué mirar sólo el porcentaje de aciertos puede engañar.', 8: '**Cómo leer la tabla y las barras de regresión.** Cada grupo A–E agrega pistas al anterior y los colores distinguen Ridge y bosque. MAE es el error promedio en los días de práctica: menor es mejor. MAE CV viene de las tres rondas antiguas; MAE entrenamiento viene de días ya estudiados. No son las mismas fechas ni necesariamente la misma escala de producción: sus diferencias no prueban por sí solas sobreajuste.\n\nEn el gráfico queremos barras por debajo de la línea punteada de la referencia. No ocurre en esta ejecución: el mejor supervisado es Ridge con historia, con MAE de 127,04, frente a 95,49 de repetir hoy. Agregar sensores y gas/agua no lo mejora en esta comparación. Conservamos la regla sencilla; no significa que esos datos nunca puedan servir en otro planteo.', 10: '**Cómo leer la tabla y las barras de clasificación.** AP indica qué tan bien quedan las caídas reales entre los casos de mayor puntuación. Aquí una barra más alta es mejor; la línea punteada es la referencia de aproximadamente 0,126. El mayor valor es 0,341 para el bosque con el grupo E, que incluye gas y agua conocidos.\n\nAP de 0,341 no significa 34,1 % de aciertos. En la tabla, precision, recall y F1 son cero porque con el corte inicial de 0,50 ninguno de estos modelos genera alarmas en validación. Pueden ordenar los riesgos de forma diferente aunque ninguna puntuación alcance 0,50. La siguiente celda elige otro corte con validación, sin tocar prueba.\n\nGanar con el grupo E no demuestra que cada sensor sea necesario ni que el gas cause cambios: estamos comparando grupos y modelos completos.', 12: '**Qué decisión salió de este bloque.** Para cantidades elegimos la referencia sencilla; Ridge queda como el mejor supervisado comparado, pero no como ganador. Para caídas elegimos el bosque con grupo E.\n\nUmbral es el punto a partir del cual avisamos caída. Con 0,10 el modelo encuentra muchas caídas de validación, pero sólo cerca de 13 de cada 100 alarmas son correctas. Con 0,20 encuentra alrededor del 36 % de las caídas y cerca de 29 de cada 100 alarmas son correctas; ese corte da el mejor F1 de los probados, aproximadamente 0,323. Con 0,50 no avisa ninguna.\n\nEl 0,20 es una puntuación usada para activar la alarma; no garantiza un 20 % de probabilidad real. Tampoco es el mismo 20 % que define cuánto debe bajar el petróleo. Son dos números con funciones distintas. Dejamos las decisiones cerradas antes de mirar prueba.', 14: '**Cómo leer las tablas del examen.** REFERENCIA es la regla sencilla; SUPERVISADO, el mejor candidato elegido en práctica; ELEGIDO, lo que decidimos usar. Algunas filas repiten números porque ELEGIDO puede ser la referencia o el supervisado. Los casilleros vacíos corresponden a medidas que no aplican a esa tarea.\n\nEn cantidades, la referencia tiene MAE 53,19 y Ridge 92,63 en prueba: sigue ganando la referencia. Que el error sea menor que en validación no demuestra que el modelo haya aprendido más: estamos mirando otro período.\n\nEn caídas, el elegido tiene AP 0,215 frente a 0,118 de la referencia. Detecta aproximadamente 27 de cada 100 caídas y cerca de 32 de cada 100 alarmas son correctas. Es una mejora para ordenar casos, pero todavía deja muchas caídas sin detectar.\n\nLa tabla por pozo importa: 44 de las 45 caídas detectadas son de F-1 C y una de F-11. Con el corte elegido no detecta caídas de F-12, F-14 ni F-15 D. El promedio general oculta esa diferencia. Los cinco pozos sí tienen ejemplos de prueba; una lista vacía en Pozos sin casos no significa cero pozos evaluados.', 16: '**Qué nos dice este gráfico.** De izquierda a derecha está el petróleo que realmente se observó; la altura muestra lo que estimamos. Sobre la diagonal acertaríamos exactamente. Un punto por encima significa que estimamos de más; por debajo, de menos. El color sólo identifica al pozo.\n\nEl gráfico muestra ELEGIDO, que en regresión es repetir el valor de hoy, no Ridge. Resume 1.711 ejemplos de prueba y su error promedio es 53,19. Ese número no es un porcentaje ni un máximo: algunos días tienen errores mucho mayores. No alcanza con que muchos puntos se vean cerca de la línea para afirmar que el método funciona igual en todos los pozos.', 18: '**Leamos los cuatro casilleros.** Arriba a la izquierda hay 1.132 días sin caída en los que no avisamos. Arriba a la derecha, 95 falsas alarmas. Abajo a la izquierda, 119 caídas que se escaparon. Abajo a la derecha, 45 caídas detectadas.\n\nEn total hubo 164 caídas y encontramos 45. Lanzamos 140 alarmas y sólo 45 coincidieron con una caída real. El casillero grande de días sin caída no debe distraernos: todavía se escapan la mayoría de las caídas. Los colores muestran cantidad de días, no probabilidades ni causas.', 20: '**Con qué nos quedamos.** Ya completamos una primera comparación de regresión y clasificación, con ajustes elegidos antes del examen. Más complejo no significó mejor para estimar cantidades. Para caídas hay una mejora parcial, concentrada en un pozo, que todavía no justifica usar estas alarmas para operar.\n\nLos CSV permiten revisar cada predicción y el HTML reúne tablas y gráficos. No significan que exista un servicio funcionando ni que se hayan guardado modelos listos para producción. El siguiente paso es estudiar los errores y la estabilidad por pozo; cualquier cambio será un nuevo experimento, sin volver a elegir mirando este mismo examen.'}
def lectura_html(numero):
    from html import escape
    texto = lecturas_resultados[numero]
    return "<div style='max-width:1000px;line-height:1.6'>" + "".join(
        "<p>" + escape(parrafo).replace("**", "") + "</p>" for parrafo in texto.split("\n\n")
    ) + "</div>"

html = ["<!doctype html><html lang='es'><meta charset='utf-8'><title>Modelos de petróleo a 7 días</title><body>",
        "<h1>Regresión y clasificación a siete días</h1>",
        "<p>Pronóstico al cierre del día con datos ya disponibles. Volumen diario: unidad pendiente. Caída: al menos 20 % con petróleo de partida positivo. Sin recomendaciones de operación.</p>",
        "<p>Ajustes elegidos con rondas antiguas; familia y grupo con validación; prueba posterior sólo al cerrar las decisiones. El historial completo ya fue explorado antes de modelar.</p>",
        "<p>MAE/RMSE: menos error es mejor. AP: mayor indica mejor orden de caídas, pero depende de su frecuencia. Precision: alarmas correctas. Recall: caídas encontradas. F1: resumen de ambas.</p>",
        "<h2>Elección cerrada antes de prueba</h2><pre>" + escape(json.dumps(seleccion, ensure_ascii=False, indent=2)) + "</pre>",
        "<h2>Validación de regresión</h2>", validacion_reg.to_html(index=False), lectura_html(8),
        "<h2>Validación de clasificación</h2>", validacion_clf.to_html(index=False), lectura_html(10),
        "<h2>Examen final</h2>", "<p>" + escape(resumen_clase) + "</p>", tabla_prueba.to_html(index=False), "<h2>Por pozo</h2>", tabla_pozos.to_html(index=False), lectura_html(14)]
for i, fig in enumerate([fig_reg, fig_clf, fig_cantidades, fig_confusion]):
    html.append(fig.to_html(full_html=False, include_plotlyjs=(i == 0)))
    html.append(lectura_html([8, 10, 16, 18][i]))
html.append(lectura_html(20))
html.append("</body></html>")
(RESULTADOS / "04_modelos.html").write_text("\n".join(html), encoding="utf-8")
print("Resultados guardados en", RESULTADOS)

**Regresión, examen final:** MAE elegido = 53.188; referencia = 53.188. La elección no mejora esa medida frente a la referencia en este período. No cambiamos el modelo después de ver este resultado.

**Clasificación, examen final:** AP elegido = 0.215; referencia = 0.118. La elección mejora esa medida en este período. No cambiamos el modelo después de ver este resultado.

**En palabras simples:** De 164 caídas reales de prueba, encontramos 45 y se escaparon 119. Hubo 95 falsas alarmas. Una mejor AP no significa que encontremos todas las caídas.

Resultados guardados en c:\Users\agusa\Desktop\ejercicios-facu\TP-FINAL_YPF\reports\modelos_7d


**Con qué nos quedamos.** Ya completamos una primera comparación de regresión y clasificación, con ajustes elegidos antes del examen. Más complejo no significó mejor para estimar cantidades. Para caídas hay una mejora parcial, concentrada en un pozo, que todavía no justifica usar estas alarmas para operar.

Los CSV permiten revisar cada predicción y el HTML reúne tablas y gráficos. No significan que exista un servicio funcionando ni que se hayan guardado modelos listos para producción. El siguiente paso es estudiar los errores y la estabilidad por pozo; cualquier cambio será un nuevo experimento, sin volver a elegir mirando este mismo examen.

## Límites que acompañan los resultados

Son un primer experimento con cinco pozos del mismo campo, no una prueba de que funcione en otro campo o un pozo nuevo. Las reglas de calidad apartan algunos casos, y las unidades/sensores siguen teniendo dudas. Clasificar una caída no explica su causa.

No estimamos intervalos de confianza, valor económico ni una regla ACT / INVESTIGATE / ABSTAIN. Todavía falta evaluar estabilidad temporal y si las puntuaciones de clasificación están bien calibradas.

Documentación de las herramientas: [preparación sin fuga de información](https://scikit-learn.org/stable/common_pitfalls.html), [búsqueda de ajustes](https://scikit-learn.org/stable/modules/grid_search.html), [datos faltantes](https://scikit-learn.org/stable/modules/impute.html).